# Data Extraction

This notebook extracts structured tabular data from the raw JSON files collected in `01_collection.ipynb`. Each raw file contains a GeoJSON response from the IEM Cow (COW) API for a single NWS Weather Forecast Office and calendar year. The extraction process flattens those responses into two analysis-ready tables: `events`, containing one row per warning event, and `stormreports`, containing one row per Local Storm Report (LSR). Both tables are written as CSV files to `data/02_extraction/`, which serves as an immutable checkpoint of the source data before any cleaning or transformation is applied. No analytical decisions are made here; the output reflects the content of the source data as faithfully as possible.

## What's in a COW file

Each `{WFO}_{YEAR}.json` is one IEM Cow (COW) verification run for a single NWS Weather Forecast Office and calendar year, scoped to tornado (TO), severe thunderstorm (SV), and flash flood (FF). The response has five top-level keys:

| Key | Contents |
|---|---|
| `generated_at` | Timestamp of the API run |
| `params` | The matching rules used: `lsrbuffer`, `wind`, `hailsize`, `warningbuffer`, `enable_shared_border`, and the `begints`/`endts` window. These are the space and time tolerances that must be identical across all years for the verification to be comparable; recording them per file makes that a checkable fact rather than an assumption. |
| `stats` | File-level pre-computed verification metrics (POD, FAR, CSI, lead-time summaries). Useful as a sanity check against the row-level tables, not for modeling. |
| `events` | A GeoJSON `FeatureCollection` with one feature per issued warning. This is the warnings table. |
| `stormreports` | A GeoJSON `FeatureCollection` with one feature per Local Storm Report (LSR). This is the LSR table. |

The two FeatureCollections map directly onto the analysis metrics:

| Metric | Source table | Field |
|---|---|---|
| Detection (POD) | `stormreports` | `warned` (bool): was the observed event warned |
| False alarms (FAR) | `events` | `verify` (bool): did the warning verify against an LSR |
| Lead time | `stormreports` (warned only) | `leadtime` (minutes): advance warning, null when `warned` is false |
| Event type | both | `lsrtype` on reports, `phenomena` on events (TO / SV / FF) |
| Year | both | `year` (calendar year) |

Lead time is carried on the storm report (`leadtime`), not on the warning, so the lead-time analysis is built from `stormreports` rows where `warned` is true. The warning-side `lead0` field exists but is frequently null.

Source: https://mesonet.agron.iastate.edu/cow/

## Implementation

Extraction is implemented by `COWExtractor` in `src/extraction/extractor.py`. It reads every `{WFO}_{YEAR}.json` file from `data/01_collection/COW/`, flattens the GeoJSON feature properties into rows, and writes two CSV files to `data/02_extraction/` as an immutable checkpoint before any cleaning decisions are applied.

| Method | Output |
|---|---|
| `extract_events()` | One row per warning event; `wfo` and `year` derived from filename |
| `extract_stormreports()` | One row per LSR; `warned=False` rows are unwarned storm reports (misses) |

Geometry is excluded from both tables because the analysis is statistical, not spatial. API-supplied `wfo` and `year` fields are replaced by filename-derived values to ensure consistency across all files.

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from extraction import COWExtractor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: raw JSON files written by 01_collection.ipynb
COW_DIR = Path("../data/01_collection/COW")

# Output: flattened CSV files, immutable checkpoint before cleaning.
# Cleaning decisions are applied in 03_cleaning.ipynb → data/03_cleaning/
EXTRACTED_DIR = Path("../data/02_extraction")

## Events

One row per warning event issued by a WFO. Key fields for analysis: `phenomena`, `issue`, `expire`, `verify`, and `lead0`. The `lead0` column is null for unverified events (i.e., those with no matching LSR).

In [2]:
extractor = COWExtractor(raw_dir=COW_DIR, extracted_dir=EXTRACTED_DIR)
print(extractor)

COWExtractor(raw=../data/01_collection/COW, extracted=../data/02_extraction)


In [3]:
events = extractor.extract_events()
print(f"{len(events):,} rows × {len(events.columns)} columns")
events.head()

08:31:42  INFO     Extracting events from ABQ_2016.json ...
08:31:42  INFO     Extracting events from ABQ_2017.json ...
08:31:42  INFO     Extracting events from ABQ_2018.json ...
08:31:42  INFO     Extracting events from ABQ_2019.json ...
08:31:42  INFO     Extracting events from ABQ_2020.json ...
08:31:42  INFO     Extracting events from ABQ_2021.json ...
08:31:42  INFO     Extracting events from ABQ_2022.json ...
08:31:42  INFO     Extracting events from ABQ_2023.json ...
08:31:42  INFO     Extracting events from ABQ_2024.json ...
08:31:42  INFO     Extracting events from ABQ_2025.json ...
08:31:42  INFO     Extracting events from ABQ_2026.json ...
08:31:42  INFO     Extracting events from ABR_2016.json ...
08:31:42  INFO     Extracting events from ABR_2017.json ...
08:31:42  INFO     Extracting events from ABR_2018.json ...
08:31:42  INFO     Extracting events from ABR_2019.json ...
08:31:42  INFO     Extracting events from ABR_2020.json ...
08:31:42  INFO     Extracting events fro

275,276 rows × 32 columns


,wfo,year,phenomena,eventid,sharedborder,issue,expire,statuses,fcster,svr_tornado_possible,...,stormreports,stormreports_all,verify,tor_in_svrtorpossible,lead0,areaverify,visual_imgurl,product_text,product_href,link
0,ABQ,2016,SV,1,65957.024837,2016-04-15T22:54:00Z,2016-04-15T23:45:00Z,EXP,43,False,...,1,1,True,False,31.0,496.389007,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2016-...
1,ABQ,2016,SV,2,84419.311608,2016-04-15T23:49:00Z,2016-04-16T00:30:00Z,EXP,GUYER,False,...,,,False,False,NaN,0.000000,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2016-...
2,ABQ,2016,SV,3,47991.747618,2016-04-16T03:50:00Z,2016-04-16T04:30:00Z,EXP,40,False,...,,,False,False,NaN,0.000000,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2016-...
3,ABQ,2016,SV,4,19922.950393,2016-04-29T03:37:00Z,2016-04-29T04:00:00Z,CON,SHOEMAKE,False,...,,,False,False,NaN,0.000000,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2016-...
4,ABQ,2016,SV,5,40443.126102,2016-04-29T07:52:00Z,2016-04-29T08:30:00Z,EXP,KJ,False,...,,,False,False,NaN,0.000000,https://mesonet.agron.iastate.edu/GIS/radmap.p...,https://mesonet.agron.iastate.edu/api/1/nwstex...,https://mesonet.agron.iastate.edu/p.php?pid=20...,https://mesonet.agron.iastate.edu/vtec/f/2016-...


In [4]:
extractor.save(events, "events.csv")

08:32:08  INFO     Saved 275,276 rows to ../data/02_extraction/events.csv


PosixPath('../data/02_extraction/events.csv')

In [5]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 275276 entries, 0 to 275275
Data columns (total 32 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   wfo                    275276 non-null  str    
 1   year                   275276 non-null  int64  
 2   phenomena              275276 non-null  str    
 3   eventid                275276 non-null  int64  
 4   sharedborder           275254 non-null  float64
 5   issue                  275276 non-null  str    
 6   expire                 275276 non-null  str    
 7   statuses               275276 non-null  str    
 8   fcster                 271605 non-null  object 
 9   svr_tornado_possible   275276 non-null  bool   
 10  significance           275276 non-null  str    
 11  hailtag                233243 non-null  object 
 12  windtag                205847 non-null  object 
 13  carea                  275276 non-null  float64
 14  ar_ugc                 275276 non-null  object 

## Storm Reports

One row per Local Storm Report (LSR). The `warned` boolean indicates whether a warning was in effect at the time of the report. Warned reports link back to the events table via the `events` foreign key column; unwarned reports (`warned=False`) have null `events` and `leadtime` values and represent misses in POD calculations.

In [6]:
stormreports = extractor.extract_stormreports()
print(f"{len(stormreports):,} rows × {len(stormreports.columns)} columns")
stormreports.head()

08:32:08  INFO     Extracting stormreports from ABQ_2016.json ...
08:32:08  INFO     Extracting stormreports from ABQ_2017.json ...
08:32:08  INFO     Extracting stormreports from ABQ_2018.json ...
08:32:08  INFO     Extracting stormreports from ABQ_2019.json ...
08:32:08  INFO     Extracting stormreports from ABQ_2020.json ...
08:32:08  INFO     Extracting stormreports from ABQ_2021.json ...
08:32:09  INFO     Extracting stormreports from ABQ_2022.json ...
08:32:09  INFO     Extracting stormreports from ABQ_2023.json ...
08:32:09  INFO     Extracting stormreports from ABQ_2024.json ...
08:32:09  INFO     Extracting stormreports from ABQ_2025.json ...
08:32:09  INFO     Extracting stormreports from ABQ_2026.json ...
08:32:09  INFO     Extracting stormreports from ABR_2016.json ...
08:32:09  INFO     Extracting stormreports from ABR_2017.json ...
08:32:09  INFO     Extracting stormreports from ABR_2018.json ...
08:32:09  INFO     Extracting stormreports from ABR_2019.json ...
08:32:09  

399,880 rows × 19 columns


,wfo,year,valid,type,magnitude,city,county,state,source,remark,typetext,lon0,lat0,events,tdq,warned,leadtime,lsrtype,link
0,ABQ,2016,2016-02-22T21:30:00Z,G,70.0,28 SE BOSQUE DEL APACHE,SOCORRO,NM,MESONET,WHITE SANDS MISSILE RANGE - ZUMWALT TRACK.,TSTM WND GST,-106.60,33.49,,False,False,NaN,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/201...
1,ABQ,2016,2016-04-15T23:25:00Z,H,1.25,3 E SENECA,UNION,NM,TRAINED SPOTTER,NaN,HAIL,-103.07,36.63,2016ABQ1SVW1,False,True,31.0,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/201...
2,ABQ,2016,2016-04-22T21:55:00Z,G,59.0,3 WSW BELEN,VALENCIA,NM,AWOS,KE80 AWOS.,TSTM WND GST,-106.83,34.65,,False,False,NaN,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/201...
3,ABQ,2016,2016-04-23T01:00:00Z,G,60.0,2 ENE CLAYTON,UNION,NM,ASOS,KCAO ASOS.,TSTM WND GST,-103.14,36.46,,False,False,NaN,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/201...
4,ABQ,2016,2016-04-24T00:19:00Z,G,68.0,3 NNE LA CIENEGA,SANTA FE,NM,ASOS,KSAF ASOS.,TSTM WND GST,-106.11,35.60,,False,False,NaN,SV,https://mesonet.agron.iastate.edu/lsr/#ABQ/201...


In [7]:
extractor.save(stormreports, "stormreports.csv")

08:32:36  INFO     Saved 399,880 rows to ../data/02_extraction/stormreports.csv


PosixPath('../data/02_extraction/stormreports.csv')

In [8]:
stormreports.info()

<class 'pandas.DataFrame'>
RangeIndex: 399880 entries, 0 to 399879
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   wfo        399880 non-null  str    
 1   year       399880 non-null  int64  
 2   valid      399880 non-null  str    
 3   type       399880 non-null  str    
 4   magnitude  134507 non-null  object 
 5   city       399880 non-null  str    
 6   county     399880 non-null  str    
 7   state      399880 non-null  str    
 8   source     399880 non-null  str    
 9   remark     348824 non-null  object 
 10  typetext   399880 non-null  str    
 11  lon0       399880 non-null  float64
 12  lat0       399880 non-null  float64
 13  events     399880 non-null  str    
 14  tdq        399880 non-null  bool   
 15  warned     399880 non-null  bool   
 16  leadtime   306042 non-null  object 
 17  lsrtype    399880 non-null  str    
 18  link       399880 non-null  str    
dtypes: bool(2), float64(2), int64(1), 

## Storm Report Counts by Year and Type

In [9]:
counts = (
    stormreports
    .groupby(["year", "lsrtype"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"FF": "Flash Flood (FF)", "SV": "Severe Thunderstorm (SV)", "TO": "Tornado (TO)"})
)
counts["Total"] = counts.sum(axis=1)
counts.loc["Total"] = counts.sum()
counts

lsrtype,Flash Flood (FF),Severe Thunderstorm (SV),Tornado (TO),Total
year,,,,
2016,4619,27411,1440,33470
2017,4954,30032,1976,36962
2018,5920,25124,1435,32479
2019,5243,31235,2099,38577
2020,4487,31471,1574,37532
2021,6494,24950,1678,33122
2022,4247,28414,1682,34343
2023,4529,37893,1778,44200
2024,6552,34240,2262,43054
